In [ ]:
import os
import shutil
import time
from pathlib import Path

import yaml
from dotenv import load_dotenv
from huggingface_hub import logging as hf_logging
from huggingface_hub import login, snapshot_download
from huggingface_hub.utils import enable_progress_bars
from tqdm.auto import tqdm


def format_size(num_bytes: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    value = float(num_bytes)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{value:.2f} {unit}"
        value /= 1024
    return f"{num_bytes} B"

/home/majestic/projects/OilOracle/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
load_dotenv()

marker = Path("configs/config.yaml")
project_root = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / marker).exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError
("Could not find configs/config.yaml from the current working directory.")
config_path = Path(os.getenv("CONFIG_PATH", project_root / "configs" / "config.yaml"))
if not config_path.is_absolute():
    config_path = project_root / config_path

with config_path.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

In [ ]:
dataset_cfg = config["news-dataset"]
raw_repo_ids = dataset_cfg.get("ids", dataset_cfg.get("id"))
if isinstance(raw_repo_ids, str):
    repo_ids = [raw_repo_ids]
elif isinstance(raw_repo_ids, list) and all(isinstance(item, str) for item in raw_repo_ids):
    repo_ids = raw_repo_ids
else:
    raise ValueError
("news-dataset.id must be a string or news-dataset.ids must be a list of strings")
output_dir = Path(dataset_cfg.get("output_dir", "data/raw"))
if not output_dir.is_absolute():
    output_dir = project_root / output_dir
output_dir.mkdir(parents=True, exist_ok=True)

hf_token = os.getenv("HF_PERSONAL_TOKEN")
if not hf_token:
    raise RuntimeError("HF_PERSONAL_TOKEN is not set in the environment.")

login(token=hf_token, add_to_git_credential=False)
enable_progress_bars()
hf_logging.set_verbosity_info()

allow_patterns = ["data/**"]
max_workers = int(os.getenv("HF_DOWNLOAD_WORKERS", "4"))

print(f"Preparing to download {len(repo_ids)} dataset(s)")
print(f"Only downloading: {allow_patterns}")
print(f"Output directory: {output_dir.resolve()}")
print(f"Workers: {max_workers}")

downloaded_snapshots = []
for index, repo_id in enumerate(repo_ids, start=1):
    print(f"\n[{index}/{len(repo_ids)}] Dataset: {repo_id}")

    dry_run_files = snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        allow_patterns=allow_patterns,
        token=hf_token,
        dry_run=True,
    )

    total_files = len(dry_run_files)
    download_files = [f for f in dry_run_files if f.will_download]
    cached_files = [f for f in dry_run_files if f.is_cached]
    download_bytes = sum(f.file_size for f in download_files)
    total_bytes = sum(f.file_size for f in dry_run_files)

    print(f"Files matched: {total_files}")
    print(f"Will download now: {len(download_files)} files ({format_size(download_bytes)})")
    print(f"Already cached: {len(cached_files)} files")
    print(f"Total matched size: {format_size(total_bytes)}")

    largest_files = sorted(download_files, key=lambda f: f.file_size, reverse=True)[:10]
    if largest_files:
        print("Largest files to download:")
        for file_info in largest_files:
            print(f"- {file_info.filename} ({format_size(file_info.file_size)})")

    print("Starting download with live progress (speed + ETA)...")
    start_time = time.time()
    repo_local_path = Path(
        snapshot_download(
            repo_id=repo_id,
            repo_type="dataset",
            allow_patterns=allow_patterns,
            token=hf_token,
            max_workers=max_workers,
            tqdm_class=tqdm,
        )
    )
    elapsed = time.time() - start_time

    print(f"Download completed in {elapsed:.1f}s")
    if download_bytes > 0 and elapsed > 0:
        print(f"Average speed: {format_size(int(download_bytes / elapsed))}/s")
    print(f"Snapshot path: {repo_local_path}")

    downloaded_snapshots.append((repo_id, repo_local_path))

Preparing to download 1 dataset(s)
Only downloading: ['data/**']
Output directory: /home/majestic/projects/OilOracle/data/raw
Workers: 4

[1/1] Dataset: Brianferrell787/financial-news-multisource


[dry-run] Fetching 131 files: 100%|██████████| 131/131 [00:03<00:00, 42.83it/s]


Files matched: 131
Will download now: 0 files (0.00 B)
Already cached: 131 files
Total matched size: 19.94 GB
Starting download with live progress (speed + ETA)...


Fetching 131 files: 100%|██████████| 131/131 [00:05<00:00, 22.06it/s]

Download completed in 6.2s
Snapshot path: /home/majestic/.cache/huggingface/hub/datasets--Brianferrell787--financial-news-multisource/snapshots/c25780f336280adb57c64bda7aed605d065c672d


In [ ]:
all_moved_folders = []
for dataset_index, (repo_id, repo_local_path) in enumerate(downloaded_snapshots, start=1):
    source_data_dir = repo_local_path / "data"
    if not source_data_dir.exists():
        raise FileNotFoundError(f"Folder not found in dataset snapshot: {source_data_dir}")
    subdirs = sorted([p for p in source_data_dir.iterdir() if p.is_dir()])
    print(f"\n[{dataset_index}/{len(downloaded_snapshots)}] {repo_id}: found {len(subdirs)} folder(s) in {source_data_dir}")
    moved_folders = []
    for idx, subdir in enumerate(subdirs, start=1):
        destination = output_dir / subdir.name
        if destination.exists():
            shutil.rmtree(destination)

        shutil.copytree(subdir, destination, symlinks=False)
        parquet_count = len(list(destination.rglob("*.parquet")))
        moved_folders.append(destination)
        all_moved_folders.append(destination)
        print(f"  [{idx}/{len(subdirs)}] moved {subdir.name} ({parquet_count} parquet file(s))")

    print(f"{repo_id}: moved {len(moved_folders)} folder(s)")

print(f"\nDone. Moved {len(all_moved_folders)} folder(s) to {output_dir.resolve()}")


[1/1] Brianferrell787/financial-news-multisource: found 24 folder(s) in /home/majestic/.cache/huggingface/hub/datasets--Brianferrell787--financial-news-multisource/snapshots/c25780f336280adb57c64bda7aed605d065c672d/data
  [1/24] moved all_the_news_2 (4 parquet file(s))
  [2/24] moved american_news_jonasbecker (1 parquet file(s))
  [3/24] moved benzinga_6000stocks (4 parquet file(s))
  [4/24] moved bloomberg_reuters (17 parquet file(s))
  [5/24] moved cnbc_headlines (1 parquet file(s))
  [6/24] moved djia_stock_headlines (1 parquet file(s))
  [7/24] moved finsen_us_2007_2023 (1 parquet file(s))
  [8/24] moved fnspid_news (59 parquet file(s))
  [9/24] moved frontpage_news (10 parquet file(s))
  [10/24] moved gold_news_kaggle (1 parquet file(s))
  [11/24] moved headlines_10sites_2007_2022 (9 parquet file(s))
  [12/24] moved huffpost_news (1 parquet file(s))
  [13/24] moved mind_news_2019 (1 parquet file(s))
  [14/24] moved nyt_articles_2000_present (5 parquet file(s))
  [15/24] moved nyt